In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# ACB metadata readiness — measured records, precise scope

Read `sports_v9/lesson.md`. Predict the counts before running. Real source metadata is separate from the clearly marked constructed identity/count examples.

In [2]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'curriculum/catalog.json').is_file())
sys.path.insert(0, str(ROOT/'src'))
from shape_lab.acb import load_matches, chronological_split, canonical_id, team_only_total
matches = load_matches(ROOT/'sports_v9/data')
OUT = ROOT/'reports/v9/acb'
OUT.mkdir(parents=True, exist_ok=True)


## One game is one observation
Final score margin is an outcome, not a pregame input.

In [3]:
ids = [r['id'] for r in matches]
teams = {r[side]['id'] for r in matches for side in ['home_team','away_team']}
assert len(ids) == len(set(ids)) == 10
margins = {r['id']:r['home_score']-r['away_score'] for r in matches}
assert margins[114243] == 15
print('Games:', len(ids), 'distinct teams:', len(teams))
print('Final margins, not pregame features:', margins)

Games: 10 distinct teams: 17
Final margins, not pregame features: {114243: 15, 114234: -4, 114169: 3, 114099: 6, 114086: -22, 178442: -4, 179612: -7, 184439: -5, 188630: 22, 191313: -1}


## Split games, not frames
This is chronological game evaluation, not unseen-team evaluation. No model is fitted.

In [4]:
assignment = chronological_split(matches)
assert assignment == chronological_split(list(reversed(matches)))
counts = {s:sum(v==s for v in assignment.values()) for s in ['fit','validation','test']}
assert list(counts.values()) == [6,2,2]
for a,b in [('fit','validation'),('fit','test'),('validation','test')]:
    assert not ({i for i,s in assignment.items() if s==a} & {i for i,s in assignment.items() if s==b})
print(counts)
print(assignment)

{'fit': 6, 'validation': 2, 'test': 2}
{114243: 'fit', 114234: 'fit', 114169: 'fit', 114099: 'fit', 114086: 'fit', 178442: 'fit', 179612: 'validation', 184439: 'validation', 188630: 'test', 191313: 'test'}


## Constructed counterexamples — not additional observed athletes
Use IDs rather than guessing from names. Sum additive team rows, not both team rows and their total.

In [5]:
assert canonical_id(11,{11:12,12:13}) == 13
try:
    canonical_id(11,{11:12,12:11})
except ValueError as error:
    print('Expected invalid alias mapping:', error)
else:
    raise AssertionError('A cycle must be rejected.')
example=[{'team_name':'A','attempts':10},{'team_name':'B','attempts':5},{'team_name':'total','attempts':15}]
assert team_only_total(example,'attempts') == 15
assert sum(r['attempts'] for r in example) == 30
print('Correct count: 15. Naive double count: 30.')

Expected invalid alias mapping: Alias cycle; fix the identity mapping first.
Correct count: 15. Naive double count: 30.


## Record evidence and its boundaries

In [6]:
result={'observed_games':len(ids),'distinct_teams':len(teams),'split_counts':counts,
        'split_game_ids':assignment,'source_integrity':'connector Git blob and local SHA-256 verified',
        'model_trained':False,'raw_tracking_included':False,
        'inference_limit':'No season-wide, unseen-team or physical-tracking validation.'}
(OUT/'results.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result,indent=2))

{
  "observed_games": 10,
  "distinct_teams": 17,
  "split_counts": {
    "fit": 6,
    "validation": 2,
    "test": 2
  },
  "split_game_ids": {
    "114243": "fit",
    "114234": "fit",
    "114169": "fit",
    "114099": "fit",
    "114086": "fit",
    "178442": "fit",
    "179612": "validation",
    "184439": "validation",
    "188630": "test",
    "191313": "test"
  },
  "source_integrity": "connector Git blob and local SHA-256 verified",
  "model_trained": false,
  "raw_tracking_included": false,
  "inference_limit": "No season-wide, unseen-team or physical-tracking validation."
}
